# 1 Non-coding variants - source domain

- You have to download the ClinVar data from its public FTP site first.
- IMPACT reference: [Ensembl Variation](https://ensembl.org/info/genome/variation/prediction/predicted_data.html)
- All Molecular consequence (MC) in ClinVar :

|SO term|Variant type|Coding (C)/non-coding (NC)|IMPACT|
|---|---|---|---|
 |SO:0001574|splice_acceptor_variant|NC |HIGH |
 |SO:0001575|splice_donor_variant|NC | HIGH |
 |SO:0001578|stop_lost|C|HIGH |
 |SO:0001582|initiator_codon_variant|C|HIGH |
 |SO:0001583|missense_variant|C| MODERATE|
 |SO:0001587|nonsense (stop_gained)|C|	HIGH|
 |SO:0001619|non-coding_transcript_variant|NC|MODIFIER|
 |SO:0001623|5_prime_UTR_variant|NC|MODIFIER|
 |SO:0001624|3_prime_UTR_variant|NC|MODIFIER|
 |SO:0001627|intron_variant|NC| MODIFIER|
 |SO:0001819|synonymous_variant|C|LOW|
 |SO:0002073|no_sequence_alteration|No sequence alternation|MODIFIER|
 |SO:0002152|genic_downstream_transcript_variant|NC|MODIFIER|
 |SO:0002153|genic_upstream_transcript_variant|NC| 	MODIFIER|

## 1.1 Process ClinVar data

In [ ]:
import pandas as pd
import warnings
import os
warnings.filterwarnings("ignore") # Suppress all warnings

sub_dir = ['x-source-data', 'a-middleware', 'b-output', 'c-statistic']
for sdir in sub_dir:
    os.makedirs(sdir, exist_ok=True)

def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
	variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
	return variant

# 1 Read the input file
df = pd.read_csv('../dataset/x-source-data/clinvar_20250421.vcf', sep='\t', header=None, comment='#')
df.columns = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']

# 2 Select valid single-nucleotide variants
nucleotides = {'A', 'C', 'T', 'G'}
df['#CHROM'] = df['#CHROM'].astype(str) # Use string type to avoid mixed integer/string objects
df['POS'] = df['POS'].astype('Int64') # Use nullable Int64 type
chr_list = [str(i) for i in range(1, 23)] + ['X', 'Y'] # Select autosomal and sex-chromosome SNVs whose reference and alternate alleles differ
df_SNV = df[
	(df['#CHROM'].isin(chr_list)) &
	(df['POS'].notna()) &
	(df['REF'].isin(nucleotides)) &
	(df['ALT'].isin(nucleotides)) &
	(df['REF'] != df['ALT'])
]
print(f"[INFO] Number of single-nucleotide variants::  {len(df_SNV)}") # 3110520

# 3 Parse the INFO column
info_list = pd.read_csv('../dataset/x-source-data/clinVar_header_light.txt', sep='\t')['INFO'].tolist()
for info in info_list:
	df_SNV[info] = df_SNV['INFO'].str.extract(rf'{info}=([^;]+)', expand=True)

# 4 Extract MC, CLNREVSTAT, CLNSIG, and CLNSIGSCV filter by review status
df_SNV = df_SNV[['#CHROM', 'POS', 'REF', 'ALT', 'MC', 'CLNREVSTAT', 'CLNSIG', 'CLNSIGSCV']]
df_review = df_SNV[(df_SNV['CLNREVSTAT']=='criteria_provided,_multiple_submitters,_no_conflicts') \
					| (df_SNV['CLNREVSTAT']=='criteria_provided,_single_submitter')\
					| (df_SNV['CLNREVSTAT']=='reviewed_by_expert_panel')]  
df_review.drop_duplicates(inplace=True)
df_review['variant38'] = retrieve_variant(df_review, '#CHROM', 'POS', 'REF', 'ALT')
print(f"[INFO] Number of SNVs passing the review-status filter::  {len(df_review)}")
df_review.to_csv("../dataset/a-middleware/ClinVar.reviewed.txt", sep='\t', index=False)

In [ ]:
import pandas as pd
# Load SNVs passing the review-status filter
df_review = pd.read_csv("../dataset/a-middleware/ClinVar.reviewed.txt", sep='\t')
dt = {
		'splice_acceptor_variant': 4, 
		'splice_donor_variant': 4,
		'stop_lost': 4,
		'initiator_codon_variant': 4,
		'missense_variant': 3,
		'nonsense': 4,
		'non-coding_transcript_variant': 1,
		'5_prime_UTR_variant': 1,
		'3_prime_UTR_variant': 1,
		'intron_variant': 1,
		'synonymous_variant': 2,
		'genic_downstream_transcript_variant': 1,
		'genic_upstream_transcript_variant': 1,
		'no_sequence_alteration': 0
	} 

# Noncoding consequence set
target_set = {
	'splice_acceptor_variant',
	'splice_donor_variant',
	'non-coding_transcript_variant',
	'5_prime_UTR_variant',
	'3_prime_UTR_variant',
	'intron_variant',
	'genic_downstream_transcript_variant',
	'genic_upstream_transcript_variant'
}

def filter_mc(mc_str):
	"""
	Desc:
		filter molecular consequence
	"""
	if pd.isna(mc_str):
		return None
	# Remove the SO identifier and retain the consequence after the colon
	terms = [x.split("|")[-1] for x in mc_str.strip(",").split(",") if x]
	# Find all terms ranked in dt
	ranked = [(t, dt.get(t, 0)) for t in terms]
	if not ranked:
		return None
	# Find the highest rank
	max_rank = max(r for _, r in ranked)
	max_terms = [t for t, r in ranked if r == max_rank]
	# Discard the record if no highest-ranked term is in target_set
	if not any(t in target_set for t in max_terms):
		return None
	# Return all highest-ranked terms joined by commas
	return ",".join(max_terms)

# Apply to the MC column
df_review["MC_filtered"] = df_review["MC"].apply(filter_mc)
df_review_noncoding = df_review.dropna(subset=["MC_filtered"])

# Clinical significance
df_review_noncoding_benign = df_review_noncoding[df_review_noncoding['CLNSIG'].notna()\
						& ((df_review_noncoding['CLNSIG']=='Benign')\
						| (df_review_noncoding['CLNSIG']=='Likely_benign')\
						| (df_review_noncoding['CLNSIG']=='Benign/Likely_benign'))]
df_review_noncoding_pathogenic = df_review_noncoding[df_review_noncoding['CLNSIG'].notna()\
						& ((df_review_noncoding['CLNSIG']=='Pathogenic')\
						| (df_review_noncoding['CLNSIG']=='Likely_pathogenic')\
						| (df_review_noncoding['CLNSIG']=='Pathogenic/Likely_pathogenic'))]		

# Save
print(f"[INFO] Number of benign noncoding SNVs:: {len(df_review_noncoding_benign)}")
print(f"[INFO] Number of pathogenic noncoding SNVs:: {len(df_review_noncoding_pathogenic)}")
df_review_noncoding_benign.to_csv("../dataset/a-middleware/ClinVar.noncoding.benign.txt", sep='\t', index=False)
df_review_noncoding_pathogenic.to_csv("../dataset/a-middleware/ClinVar.noncoding.pathogenic.txt", sep='\t', index=False)

In [ ]:
# Collect all molecular-consequence types
all_mc = set()
df_review = df_review[df_review['MC'].notna()] 
for item in df_review['MC'].tolist():
	item_ls = item.split(",")
	for it in item_ls:
		all_mc.add(it)

## 1.2 Process HGMD data

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore") # Suppress all warnings

def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
	variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
	return variant

hgmd_usecols = ['acc_num', 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38', 'amino', 'disease', 'tag', 'mutype', 'refseq', 'hgvs', 'codon']
df = pd.read_csv("../dataset/x-source-data/HGMD_2023.3_allmut.csv", usecols=hgmd_usecols)
df['pos_hg38'] = df['pos_hg38'].astype('Int64')
df['chrom_hg38'] = df['chrom_hg38'].astype(str) 

# Non-coding SNVs
nucleotides = {'A', 'C', 'T', 'G'}
chr_list = [str(i) for i in range(1, 23)] + ['X', 'Y']
df_nc_SNV = df[
	(df['chrom_hg38'].isin(chr_list)) & # Autosomes and sex chromosomes
	(df['pos_hg38'].notna()) & # Non-null
	(df['amino']=='-') & # Not represented at the amino-acid level
	(df['ref_hg38'].isin(nucleotides)) & # ATGC
	(df['alt_hg38'].isin(nucleotides)) & # ATGC
	(df['ref_hg38'] != df['alt_hg38']) & # Reference and alternate alleles must differ
	(df['mutype']!='exonic-splice')  # Exclude exonic-splice variants
]
df_nc_SNV['variant_hg38'] = retrieve_variant(df_nc_SNV, 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38')
df_nc_SNV.drop_duplicates(subset='variant_hg38', inplace=True)  # Remove duplicates
print(f'[INFO] Number of non-coding SNVs:: {len(df_nc_SNV)}')
# DM + DM?
df_nc_SNV = df_nc_SNV[(df_nc_SNV['tag'] == 'DM') | (df_nc_SNV['tag'] == 'DM?')]
print(f'[INFO] Number classified as DM or DM?:: {len(df_nc_SNV)}')
print(df_nc_SNV['mutype'].value_counts())

# Save
df_nc_SNV.to_csv("../dataset/a-middleware/HGMD.noncodingSNV.dm+dmq.txt", sep='\t', index=False)

## 1.3 Merge datasets

In [ ]:
# Pathogenic variants
import pandas as pd
import warnings
import os
warnings.filterwarnings("ignore") # Suppress all warnings

hgmd_path = "../dataset/a-middleware/HGMD.noncodingSNV.dm+dmq.txt"
clinvar_path = "../dataset/a-middleware/ClinVar.noncoding.pathogenic.txt"

clinvar_pt = pd.read_csv(clinvar_path, sep='\t', usecols=['variant38'])
clinvar_pt['source'] = ['clinvar'] * len(clinvar_pt)

hgmd_pt = pd.read_csv(hgmd_path, sep='\t', usecols=['variant_hg38'])
hgmd_pt.columns = ['variant38']
hgmd_pt['source'] = ['hgmd'] * len(hgmd_pt)

pt_df = pd.concat([clinvar_pt, hgmd_pt], ignore_index=True)
pt_df = pt_df.groupby('variant38')['source'].agg(lambda x: ';'.join(x)).reset_index()
pt_df['label'] = [1] * len(pt_df)
print(f"[INFO] Number of pathogenic ClinVar variants: {len(clinvar_pt)}, number of pathogenic HGMD variants: {len(hgmd_pt)}, after merging:  {len(pt_df)}")

# save result
out_path = "../dataset/a-middleware/noncoding/pathogenic.txt"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
pt_df.to_csv(out_path, sep='\t', index=False)

In [ ]:
# Benign variants
clinvar_path_bn = "../dataset/a-middleware/ClinVar.noncoding.benign.txt"

clinvar_bn = pd.read_csv(clinvar_path_bn, sep='\t', usecols=['#CHROM', 'POS', 'REF', 'ALT', 'variant38'])
clinvar_bn['source'] = ['clinvar'] * len(clinvar_bn)
clinvar_bn['label'] = [0] * len(clinvar_bn)
clinvar_bn.drop_duplicates(subset='variant38', inplace=True)
clinvar_bn.to_csv("../dataset/a-middleware/noncoding/benign.txt", sep='\t', index=False)
print(f"[INFO] Number of benign ClinVar variants: {len(clinvar_bn)}")

## 1.4 Close-by

In [ ]:
def expand_df(df):
	df[['first_part', 'ALT']] = df['variant38'].str.split('/', expand=True)
	df[['#CHROM', 'POS', 'REF']] = df['first_part'].str.split('_', expand=True)
	df['POS'] = df['POS'].astype(int)
	df.drop('first_part', axis=1, inplace=True)
	df = df[['#CHROM', 'POS', 'REF', 'ALT', 'variant38', 'source', 'label']]
	return df
df_noncoding_pt = pd.read_csv("../dataset/a-middleware/noncoding/pathogenic.txt", sep='\t')
df_noncoding_pt = expand_df(df_noncoding_pt)
df_noncoding_pt.to_csv("../dataset/a-middleware/noncoding/pathogenic.txt", sep='\t', index=False)

In [ ]:
def get_closeby_pairs(df1, df2):
	"""
	Desc:
		Use close-by method to create a balanced dataset
	"""
	df1['#CHROM'] = df1['#CHROM'].astype(str)
	df2['#CHROM'] = df2['#CHROM'].astype(str)

	df1['POS'] = df1['POS'].astype(int)
	df2['POS'] = df2['POS'].astype(int)

	print(f"[INFO] Initial DataFrame: {len(df2)}")
	close_bn_set = set()
	closeby_records = []

	for _, row in df1.iterrows():
		chrom = row['#CHROM']
		pos = row['POS']
		var_pos = row['variant38']

		tmp_bn_df = df2[df2['#CHROM'] == chrom]
		if tmp_bn_df.empty:
			print(f'[Error] No opposite-label variant was found on chromosome: {chrom}')
			continue

		tmp_bn_df['abs_dist'] = abs(tmp_bn_df['POS'] - pos)
		tmp_bn_df = tmp_bn_df.sort_values(by='abs_dist', ascending=True)

		index = 0
		while index < len(tmp_bn_df):
			var_neg = tmp_bn_df.iloc[index]['variant38']
			dist = tmp_bn_df.iloc[index]['abs_dist']
			if var_neg in close_bn_set:
				index += 1
			else:
				close_bn_set.add(var_neg)
				closeby_records.append([var_pos, var_neg, dist])
				break

	closeby_df = pd.DataFrame(closeby_records, columns=['pos_variant38', 'neg_variant38', 'abs_distance'])
	return closeby_df

df_noncoding_pt = pd.read_csv("../dataset/a-middleware/noncoding/pathogenic.txt", sep='\t')
df_noncoding_bn = pd.read_csv("../dataset/a-middleware/noncoding/benign.txt", sep='\t')
closeby_df = get_closeby_pairs(df_noncoding_pt, df_noncoding_bn)
closeby_df.to_csv(f"../dataset/a-middleware/noncoding/PAIRS_noncoding.txt", sep='\t', index=False)

In [ ]:
def expand_df_2(df, vid_column):
	df[['first_part', 'ALT']] = df[vid_column].str.split('/', expand=True)
	df[['#CHROM', 'POS', 'REF']] = df['first_part'].str.split('_', expand=True)
	df['POS'] = df['POS'].astype(int)
	df.drop('first_part', axis=1, inplace=True)
	df = df[['#CHROM', 'POS', 'REF', 'ALT']]
	return df

# save result
out_path = "../dataset/b-output/noncoding_dataset/noncoding.pt.txt"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

pos_vcf = expand_df_2(closeby_df, "pos_variant38")
pos_vcf.to_csv(out_path, sep='\t', index=False)
neg_vcf = expand_df_2(closeby_df, "neg_variant38")
neg_vcf.to_csv(out_path.replace(".pt.txt", ".bn.txt"), sep='\t', index=False)

## 1.5 Remove variants with conflicting pathogenic/benign labels

In [ ]:
pairs = pd.read_csv(f"../dataset/a-middleware/noncoding/PAIRS_noncoding.txt", sep='\t')
# Find the intersection
overlap = set(pairs["pos_variant38"]) & set(pairs["neg_variant38"])
# Remove intersecting rows
df_cleaned = pairs[~pairs["pos_variant38"].isin(overlap) & ~pairs["neg_variant38"].isin(overlap)]
print(len(df_cleaned))

## 1.6 Remove target-domain data from the source domain

In [ ]:
root_path = "../dataset/b-output/synonymous_dataset/"
target_test = pd.read_csv(f"{root_path}/test_238.txt", sep='\t')['variant_hg38'].tolist()
target_train = pd.read_csv(f"{root_path}/train_2362.txt", sep='\t')['variant_hg38'].tolist()
df_cleaned = df_cleaned[~df_cleaned["pos_variant38"].isin(target_test) & ~df_cleaned["neg_variant38"].isin(target_test)]
df_cleaned = df_cleaned[~df_cleaned["pos_variant38"].isin(target_train) & ~df_cleaned["neg_variant38"].isin(target_train)]
print(len(df_cleaned))

In [ ]:
# Save the data
df_cleaned.to_csv(f"../dataset/a-middleware/noncoding/PAIRS_noncoding.cleaned.txt", sep='\t', index=False)

In [ ]:
def expand_df_2(df, vid_column):
	df[['first_part', 'ALT']] = df[vid_column].str.split('/', expand=True)
	df[['#CHROM', 'POS', 'REF']] = df['first_part'].str.split('_', expand=True)
	df['POS'] = df['POS'].astype(int)
	df.drop('first_part', axis=1, inplace=True)
	df = df[['#CHROM', 'POS', 'REF', 'ALT']]
	return df

pos_vcf = expand_df_2(df_cleaned, "pos_variant38")
pos_vcf.to_csv("../dataset/b-output/noncoding_dataset/noncoding.pt.cleaned.txt", sep='\t', index=False)
neg_vcf = expand_df_2(df_cleaned, "neg_variant38")
neg_vcf.to_csv("../dataset/b-output/noncoding_dataset/noncoding.bn.cleaned.txt", sep='\t', index=False)

## 1.7 Remove DM? records

In [ ]:
import pandas as pd

pairs_df = pd.read_csv("../dataset/a-middleware/noncoding/PAIRS_noncoding.txt", sep='\t')
pathogenic_df = pd.read_csv("../dataset/a-middleware/noncoding/pathogenic.all.mutype.source.evidence.txt", sep='\t')
print(f"[INFO] pathogenic len: {len(pathogenic_df)}")
pathogenic_df_nodmq = pathogenic_df[pathogenic_df['evidence_level']!='DM?']
print(f"[INFO] pathogenic no dmq: {len(pathogenic_df_nodmq)}")
# Remove DM? records
pairs_df = pairs_df[pairs_df['pos_variant38'].isin(pathogenic_df_nodmq['variant38'])]
pairs_df.to_csv(f"../dataset/a-middleware/noncoding/PAIRS_noncoding_noDMq.txt", sep='\t', index=False)

# 2 Missense variants - source domain


## 2.1 HGMD data

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore") # Suppress all warnings


def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
	variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
	return variant


hgmd_usecols = ['acc_num', 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38', 'amino', 'disease', 'tag', 'mutype', 'refseq', 'hgvs', 'codon']
df = pd.read_csv("../dataset/x-source-data/HGMD_2023.3_allmut.csv", usecols=hgmd_usecols)
df['pos_hg38'] = df['pos_hg38'].astype('Int64') # Nullable integer type
df['chrom_hg38'] = df['chrom_hg38'].astype(str) 
nucleotides = {'A', 'C', 'T', 'G'}
chr_list = [str(i) for i in range(1, 23)] + ['X', 'Y']

missense_df = df[
	(df['chrom_hg38'].isin(chr_list)) &
	(df['mutype'] == 'missense') &
	(df['pos_hg38'].notna()) &
	(df['ref_hg38'].isin(nucleotides)) &
	(df['alt_hg38'].isin(nucleotides)) &
	(df['ref_hg38'] != df['alt_hg38'])
]
print(f'[INFO] Number of missense variants:: {len(missense_df)}')

missense_dm_df = missense_df[(missense_df['tag'] == 'DM') | (missense_df['tag'] == 'DM?')]
print(f'[INFO] Number classified as DM or DM?:: {len(missense_dm_df)}')

missense_dm_df['variant_hg38'] = retrieve_variant(missense_dm_df, 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38')
missense_dm_df.drop_duplicates(subset='variant_hg38', inplace=True)
print(f"[INFO] Remaining after deduplication by chromosome, position, reference allele, and alternate allele:: {len(missense_dm_df)}  records")

# save result
missense_dm_df.to_csv("../dataset/a-middleware/HGMD.missense.dm+dmq.txt", sep='\t', index=False)

## 2.2 ClinVar data

In [ ]:
import pandas as pd
# Load SNVs passing the review-status filter
df_review = pd.read_csv("../dataset/a-middleware/ClinVar.reviewed.txt", sep='\t')
# Missense variants
dt = {
	'splice_acceptor_variant': 4, 
	'splice_donor_variant': 4,
	'stop_lost': 4,
	'initiator_codon_variant': 4,
	'missense_variant': 3,
	'nonsense': 4,
	'non-coding_transcript_variant': 1,
	'5_prime_UTR_variant': 1,
	'3_prime_UTR_variant': 1,
	'intron_variant': 1,
	'synonymous_variant': 2,
	'genic_downstream_transcript_variant': 1,
	'genic_upstream_transcript_variant': 1,
	'no_sequence_alteration': 0
	} 

# Missense consequence set
target_set = { 'missense_variant' }

def filter_mc(mc_str):
	"""
	Desc:
		filter molecular consequence
	"""
	if pd.isna(mc_str):
		return None
	# Remove the SO identifier and retain the consequence after the colon
	terms = [x.split("|")[-1] for x in mc_str.strip(",").split(",") if x]
	# Find all terms ranked in dt
	ranked = [(t, dt.get(t, 0)) for t in terms]
	if not ranked:
		return None
	# Find the highest rank
	max_rank = max(r for _, r in ranked)
	max_terms = [t for t, r in ranked if r == max_rank]
	# Discard the record if no highest-ranked term is in target_set
	if not any(t in target_set for t in max_terms):
		return None
	# Return all highest-ranked terms joined by commas
	return ",".join(max_terms)

# Apply to the MC column
df_review["MC_filtered"] = df_review["MC"].apply(filter_mc)
df_review_missense = df_review.dropna(subset=["MC_filtered"])

# Clinical significance
df_review_missense_benign = df_review_missense[df_review_missense['CLNSIG'].notna()\
						& ((df_review_missense['CLNSIG']=='Benign')\
						| (df_review_missense['CLNSIG']=='Likely_benign')\
						| (df_review_missense['CLNSIG']=='Benign/Likely_benign'))]
df_review_missense_pathogenic = df_review_missense[df_review_missense['CLNSIG'].notna()\
						& ((df_review_missense['CLNSIG']=='Pathogenic')\
						| (df_review_missense['CLNSIG']=='Likely_pathogenic')\
						| (df_review_missense['CLNSIG']=='Pathogenic/Likely_pathogenic'))]		

# Save
print(f"[INFO] Number of benign missense SNVs:: {len(df_review_missense_benign)}")
print(f"[INFO] Number of pathogenic missense SNVs:: {len(df_review_missense_pathogenic)}")
df_review_missense_benign.to_csv(f"../dataset/a-middleware/ClinVar.missense.benign.txt", sep='\t', index=False)
df_review_missense_pathogenic.to_csv(f"../dataset/a-middleware/ClinVar.missense.pathogenic.txt", sep='\t', index=False)

## 2.3 Merge datasets

In [ ]:
# Pathogenic variants
import pandas as pd
import warnings
warnings.filterwarnings("ignore") # Suppress all warnings

hgmd_path = "../dataset/a-middleware/HGMD.missense.dm+dmq.txt"
clinvar_path = "../dataset/a-middleware/ClinVar.missense.pathogenic.txt"

clinvar_pt = pd.read_csv(clinvar_path, sep='\t', usecols=['variant38'])
clinvar_pt['source'] = ['clinvar'] * len(clinvar_pt)

hgmd_pt = pd.read_csv(hgmd_path, sep='\t', usecols=['variant_hg38'])
hgmd_pt.columns = ['variant38']
hgmd_pt['source'] = ['hgmd'] * len(hgmd_pt)

pt_df = pd.concat([clinvar_pt, hgmd_pt], ignore_index=True)
pt_df = pt_df.groupby('variant38')['source'].agg(lambda x: ';'.join(x)).reset_index()
pt_df['label'] = [1] * len(pt_df)
print(f"[INFO] Number of pathogenic ClinVar variants: {len(clinvar_pt)}, number of pathogenic HGMD variants: {len(hgmd_pt)}, after merging:  {len(pt_df)}")

# save result
out_path = "../dataset/a-middleware/missense/pathogenic.txt"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
pt_df.to_csv(out_path, sep='\t', index=False)

## 2.4 Remove variants with conflicting pathogenic/benign labels

In [ ]:
# Remove pathogenic variants with conflicting labels
pt_df = pd.read_csv("../dataset/a-middleware/missense/pathogenic.txt", sep='\t')
pt_df_clean = pt_df[~pt_df['variant38'].isin(df_review_missense_benign['variant38'])]
print(len(pt_df_clean))

In [ ]:
# Remove benign variants with conflicting labels
bn_df_clean = df_review_missense_benign[~df_review_missense_benign['variant38'].isin(pt_df['variant38'])]
print(len(bn_df_clean))

## 2.5 Remove target-domain variants from the source domain

In [ ]:
root_path = "../dataset/b-output/synonymous_dataset/"
target_test = pd.read_csv(f"{root_path}/test_238.txt", sep='\t')['variant_hg38'].tolist()
target_train = pd.read_csv(f"{root_path}/train_2362.txt", sep='\t')['variant_hg38'].tolist()
bn_df_clean = bn_df_clean[~bn_df_clean["variant38"].isin(target_test)]
pt_df_clean = pt_df_clean[~pt_df_clean["variant38"].isin(target_train)]
print(len(pt_df_clean))
print(len(bn_df_clean))

In [ ]:
# save the data
pt_df_clean.to_csv(f"../dataset/a-middleware/missense/pathogenic.clean.txt", sep='\t', index=False)
bn_df_clean.to_csv(f"../dataset/a-middleware/missense/benign.clean.txt", sep='\t', index=False)

## 2.6 Close-by

In [ ]:
pt_df_clean = pd.read_csv(f"../dataset/a-middleware/missense/pathogenic.clean.txt", sep='\t')
bn_df_clean = pd.read_csv(f"../dataset/a-middleware/missense/benign.clean.txt", sep='\t')

def expand_df(df):
	df[['first_part', 'ALT']] = df['variant38'].str.split('/', expand=True)
	df[['#CHROM', 'POS', 'REF']] = df['first_part'].str.split('_', expand=True)
	df['POS'] = df['POS'].astype(int)
	df.drop('first_part', axis=1, inplace=True)
	df = df[['#CHROM', 'POS', 'REF', 'ALT', 'variant38', 'source', 'label']]
	return df
pt_df_clean = expand_df(pt_df_clean)

In [ ]:
from datetime import datetime


def get_closeby_pairs(df1, df2):
	print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Begin to match... ")
	df1['#CHROM'] = df1['#CHROM'].astype(str)
	df2['#CHROM'] = df2['#CHROM'].astype(str)

	df1['POS'] = df1['POS'].astype(int)
	df2['POS'] = df2['POS'].astype(int)

	print(f"[INFO] Initial DataFrame: {len(df2)}")
	close_bn_set = set()
	closeby_records = []

	for _, row in df1.iterrows():
		chrom = row['#CHROM']
		pos = row['POS']
		var_pos = row['variant38']

		tmp_bn_df = df2[df2['#CHROM'] == chrom]
		if tmp_bn_df.empty:
			print(f'[Error] No opposite-label variant was found on chromosome: {chrom}')
			continue

		tmp_bn_df['abs_dist'] = abs(tmp_bn_df['POS'] - pos)
		tmp_bn_df = tmp_bn_df.sort_values(by='abs_dist', ascending=True)

		index = 0
		while index < len(tmp_bn_df):
			var_neg = tmp_bn_df.iloc[index]['variant38']
			dist = tmp_bn_df.iloc[index]['abs_dist']
			if var_neg in close_bn_set:
				index += 1
			else:
				close_bn_set.add(var_neg)
				closeby_records.append([var_pos, var_neg, dist])
				break

	closeby_df = pd.DataFrame(closeby_records, columns=['neg_variant38', 'pos_variant38', 'abs_distance'])
	print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Finished!")
	return closeby_df

In [ ]:
closeby_df = get_closeby_pairs(bn_df_clean, pt_df_clean)
closeby_df.to_csv(f"../dataset/a-middleware/missense/PAIRS_missnese.txt", sep='\t', index=False)

## 2.7 Save pathogenic and benign variant information separately 

In [ ]:
def expand_df(df, label, vid_colname):
	df = df[[vid_colname]]
	df.rename(columns={vid_colname: 'variant38'}, inplace=True)
	df[['first_part', 'ALT']] = df['variant38'].str.split('/', expand=True)
	df[['#CHROM', 'POS', 'REF']] = df['first_part'].str.split('_', expand=True)
	df['POS'] = df['POS'].astype(int)
	df.drop('first_part', axis=1, inplace=True)
	df['label'] = [label] * len(df)
	df = df[['#CHROM', 'POS', 'REF', 'ALT', 'variant38', 'label']]
	return df

In [ ]:
pt_expand = expand_df(df=closeby_df, label=1, vid_colname='pos_variant38')
pt_expand.to_csv(f"../dataset/b-output/missense_dataset/pathogenic.rmSyn.txt", sep='\t', index=False)
bn_expand = expand_df(df=closeby_df, label=0, vid_colname='neg_variant38')
bn_expand.to_csv(f"../dataset/b-output/missense_dataset/benign.rmSyn.txt", sep='\t', index=False)

## 2.8 Add transcript and CDS-position information 

### (A) Obtain CDS and transcript information from HGMD

In [ ]:
import pandas as pd

# Load missense-variant information
df_missense_pt = pd.read_csv("../dataset/b-output/missense_dataset/pathogenic.rmSyn.txt", sep='\t')
print(f"[INFO] Missense_pt (initial): {len(df_missense_pt)}...")
df_missense_pt.drop_duplicates(subset='variant38', inplace=True)
print(f"[INFO] Missense_pt (drop dup): {len(df_missense_pt)}...")

# Load HGMD information
usecols = ['disease', 'amino', 'codon', 'refseq', 'hgvs', 'variant_hg38']
hgmd_missense_df = pd.read_csv("../dataset/a-middleware/HGMD.missense.dm+dmq.txt", sep='\t', usecols=usecols)
hgmd_missense_df.rename(columns={'variant_hg38': 'variant38', 'codon': 'aa_pos'}, inplace=True)
hgmd_missense_df.drop_duplicates(subset='variant38', inplace=True)
print(f"[INFO] Missense_pt: {len(hgmd_missense_df)}...")

# Merge annotation information
df_missense_pt = df_missense_pt.merge(hgmd_missense_df, how='left', left_on='variant38', right_on='variant38', suffixes=('', '_m'))
print(f"[INFO] After merge: {len(df_missense_pt)}...")

In [ ]:
# Records without annotations
df_missense_hit = df_missense_pt[df_missense_pt['refseq'].notna()]
df_missense_left = df_missense_pt[~df_missense_pt['refseq'].notna()]
print(f"[INFO] Missense without refseq anno: {len(df_missense_left)}")
df_missense_left.to_csv("../dataset/a-middleware/missense/Clinvar.pt.sole.txt", sep='\t', index=False)

# Standardize columns for annotated records
df_missense_hit[['cds_pos', 'allele_change']] = df_missense_hit['hgvs'].str.extract(r'(\d+)([ACGT]>[ACGT])') # Extract cds_pos with a regular expression
df_missense_hit.rename(columns={'codon': 'aa_pos'}, inplace=True) # Rename columns
df_missense_hit.drop('hgvs', axis=1, inplace=True) # Drop redundant columns
df_missense_hit['aa_pos'] = df_missense_hit['aa_pos'].astype('Int64') # Nullable integer type
df_missense_hit['cds_pos'] = df_missense_hit['cds_pos'].astype('Int64')
df_missense_hit = df_missense_hit[['#CHROM', 'POS', 'REF', 'ALT', 'variant38', 'label', 'source', 'CLNSIG', 'tag', 'evidence_level', 'refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos', 'disease']]
df_missense_hit.to_csv(f"../dataset/a-middleware/missense/HGMD.pt.sole.annotated.txt", sep='\t', index=False)

### (B) Obtain CDS and transcript information from ClinVar

ClinVar is distributed in two formats: `.vcf` (`clinvar_YYYYMMDD.vcf.gz`; see the [reference](https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/)) and tab-delimited `.txt` (`variant_summary_YYYY-MM.txt.gz`). The VCF follows VCF4 strictly and is updated more frequently, whereas the TXT file includes transcript information.

In [ ]:
import re

def parse_variant(s):
	pattern = re.compile(
		r'(?P<refseq>NM_[0-9.]+)'        # NM accession
		r'(?:\([A-Za-z0-9_]+\))?'        # optional gene-name component
		r':c\.(?P<cds_pos>\d+)'          # CDS position
		r'(?P<allele_change>[ACGT]>[ACGT])'  # nucleotide change
		r'(?:\s*\(p\.(?P<aa_ref>[A-Za-z]{3})(?P<aa_pos>\d+)(?P<aa_alt>[A-Za-z]{3})\))?'  # amino-acid change
	)
	m = pattern.search(s)
	if not m:
		return pd.Series([None]*5, index=['refseq','cds_pos','allele_change','amino','aa_pos'])
	refseq = m.group('refseq')
	cds_pos = m.group('cds_pos')
	allele_change = m.group('allele_change')
	if m.group('aa_ref') and m.group('aa_alt'):
		amino = f"{m.group('aa_ref')}-{m.group('aa_alt')}"
		aa_pos = m.group('aa_pos')
	else:
		amino, aa_pos = None, None
	return pd.Series([refseq, cds_pos, allele_change, amino, aa_pos],
					 index=['refseq','cds_pos','allele_change','amino','aa_pos'])

In [ ]:
# 1 Load ClinVar information
df_clinvar = pd.read_csv("../dataset/x-source-data/variant_summary_2025-04.GRCh38.txt", sep='\t')

# 2 Pathogenic variants: add records not annotated by HGMD
df_mis_pt = pd.read_csv("../dataset/a-middleware/missense/Clinvar.pt.sole.txt", sep='\t')
df_mis_pt.drop(['disease', 'amino', 'codon', 'refseq', 'hgvs'], axis=1, inplace=True) # Remove obsolete information columns
hit_df = df[df['variant38'].isin(df_mis_pt['variant38'])] # Select records matching variant38
hit_df_2 = pd.concat([hit_df, hit_df['Name'].apply(parse_variant)], axis=1) # Extract RefSeq and related information
hit_df_2 = hit_df_2[['variant38', 'refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos', 'PhenotypeIDS']] # Retain annotation columns
hit_df_2.rename(columns={'PhenotypeIDS': 'disease'}, inplace=True) # Rename columns
df_mis_pt = df_mis_pt.merge(hit_df_2, how='left', left_on='variant38', right_on='variant38', suffixes=('', '_m')) # merge
df_mis_pt.dropna(subset=['refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos'], inplace=True) # Remove rows without transcript or CDS information
df_mis_pt.to_csv(f"../dataset/a-middleware/missense/Clinvar.pt.sole.annotated.txt", sep='\t', index=False)

In [ ]:
# 3 Benign variants from ClinVar only
df_missense_bn = pd.read_csv("../dataset/b-output/missense_dataset/benign.rmSyn.txt", sep='\t')
hit_df = df_clinvar[df_clinvar['variant38'].isin(df_missense_bn['variant38'])]
hit_df_2 = pd.concat([hit_df, hit_df['Name'].apply(parse_variant)], axis=1)
hit_df_2 = hit_df_2[['variant38', 'refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos', 'PhenotypeIDS']] # Retain annotation columns
hit_df_2.rename(columns={'PhenotypeIDS': 'disease'}, inplace=True) # Rename columns
df_missense_bn = df_missense_bn.merge(hit_df_2, how='left', left_on='variant38', right_on='variant38', suffixes=('', '_m')) # merge
df_missense_bn.dropna(subset=['refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos'], inplace=True) # Remove rows without transcript or CDS information
df_missense_bn.to_csv(f"../dataset/a-middleware/missense/Clinvar.bn.sole.annotated.txt", sep='\t', index=False)

### (C) Filter sample pairs using the retained and close-by data

In [ ]:
# Load data
df_mis_pt_hgmd = pd.read_csv("../dataset/a-middleware/missense/Hgmd.pt.sole.annotated.txt", sep='\t')
df_mis_pt_clinvar = pd.read_csv("../dataset/a-middleware/missense/Clinvar.pt.sole.annotated.txt", sep='\t')
df_mis_pt = pd.concat([df_mis_pt_hgmd, df_mis_pt_clinvar], ignore_index=True)
df_mis_bn = pd.read_csv("../dataset/a-middleware/missense/Clinvar.bn.sole.annotated.txt", sep='\t')
df_pairs = pd.read_csv("../dataset/a-middleware/missense/PAIRS_missnese.txt", sep='\t')

In [ ]:
df_mis_pt.dropna(subset=['refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos'], inplace=True) # Remove rows without transcript or CDS information
df_mis_bn.dropna(subset=['refseq', 'cds_pos', 'allele_change', 'amino', 'aa_pos'], inplace=True) # Remove rows without transcript or CDS information

print(f"[INFO] Initial pairs: {len(df_pairs)}")
df_pairs = df_pairs[(df_pairs['neg_variant38'].isin(df_mis_bn['variant38'])) & (df_pairs['pos_variant38'].isin(df_mis_pt['variant38']))]
df_pairs.to_csv(f"../dataset/a-middleware/missense/PAIRS_missnese.sanity.check.1.txt", sep='\t', index=False)
print(f"[INFO] After remove non-info records: {len(df_pairs)}")

df_mis_bn = df_mis_bn[df_mis_bn['variant38'].isin(df_pairs['neg_variant38'])]
df_mis_bn.to_csv(f"../dataset/b-output/missense_dataset/missense.bn.with_transcript.txt", sep='\t', index=False)
df_mis_pt = df_mis_pt[df_mis_pt['variant38'].isin(df_pairs['pos_variant38'])]
df_mis_pt.to_csv(f"../dataset/b-output/dataset_missense/missense.pt.with_transcript.txt", sep='\t', index=False)

### (D) Collect transcript information 

- HGMD and ClinVar use RefSeq identifiers, whereas the previously collected MANE Select records were organized primarily by Ensembl identifiers.
- To address this issue, obtain [NCBI RefSeq information](https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/GRCh38_latest/refseq_identifiers/GRCh38_latest_rna.gbff.gz), for annotation.
- For 2,490 unmatched RefSeq IDs, GRCh38_latest_rna.gbff.gz contained no matching record; these IDs may have been suppressed or made obsolete and are therefore no longer applicable.

In [ ]:
# 1 Load transcript information
transcript_path = "../dataset/x-source-data/transcript_id_info_summary.txt"
df_transcript = pd.read_csv(transcript_path, sep='\t')
df_transcript.head()

In [ ]:
# 2 Identify RefSeq accessions without annotations
s1 = set(df_mis_bn['refseq'])
s2 = set(df_mis_pt['refseq'])
s3 = s1.union(s2)
s4 = set(df_transcript['refseq_id'])
nomatch = s3 - s4
print(f"[INFO] Unavailable refseq infos: {len(nomatch)}")
df_nomatch = pd.DataFrame({'refseq': list(nomatch)})
df_nomatch.head()

In [ ]:
# 3 Annotate RefSeq information
from Bio import SeqIO
from tqdm import tqdm
from datetime import datetime

rna_gbff = "../dataset/x-source-data/GRCh38_latest_rna.gbff"

cds_list = []
aa_list = []
refseq_list = []
strand_list = []

# Build an index
print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Building the index...")
seq_index = SeqIO.index(rna_gbff, "gb")

print(f"[INFO] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Starting the search...")

for refseq in tqdm(df_nomatch['refseq'].tolist()):
	# Attempt retrieval through the index
	try:
		record = seq_index[refseq]  # If df_nomatch lacks version suffixes, consider accession mapping
		found_cds = False
		for feature in record.features:
			if feature.type == "CDS":
				# CDS sequence
				cds_seq = feature.extract(record.seq)
				cds_list.append(str(cds_seq))

				# Amino-acid sequence
				if "translation" in feature.qualifiers:
					aa_seq = feature.qualifiers["translation"][0]
				else:
					aa_seq = str(cds_seq.translate(to_stop=True))  # Translate directly when no translation field is present
				aa_list.append(aa_seq)

				# Strand orientation
				strand = feature.location.strand  # 1 = Forward strand, -1 = Reverse strand
				strand_list.append(strand)

				# Corresponding RefSeq accession
				refseq_list.append(refseq)
				found_cds = True
		if not found_cds:
			print(f"[WARN] {refseq} has no CDS in the record")
			cds_list.append("")
			aa_list.append("")
			refseq_list.append(refseq)
			strand_list.append(None)

	except KeyError:
		print(f"[WARN] {refseq} No matching record")
		cds_list.append("")
		aa_list.append("")
		refseq_list.append(refseq)
		strand_list.append(None)


df_cds = pd.DataFrame({
	"refseq": refseq_list,
	"cds_seq": cds_list,
	"aa_seq": aa_list,
	"strand": strand_list
})

In [ ]:
# 4 Standardize the DataFrame
df_cds.dropna(inplace=True) # Remove records with missing information
df_cds.columns = ['refseq_id', 'coding_sequence', 'aa_sequence', 'strand'] # Rename columns
df_cds['transcript_id'] = [pd.NA] * len(df_cds) # Add the transcript_id column
df_cds['cds_length'] = df_cds['coding_sequence'].apply(len) # Calculate coding_sequence length for each row
df_cds['aa_length'] = df_cds['aa_sequence'].apply(len) # Calculate aa_sequence length for each row
df_cds = df_cds[['transcript_id', 'strand', 'refseq_id', 'coding_sequence', 'cds_length',  'aa_sequence', 'aa_length']] # Reorder columns

In [ ]:
# 5 Merge information
df_transcript = pd.concat([df_transcript, df_cds], ignore_index=True)
df_transcript['cds_length'] = df_transcript['cds_length'].astype('Int64')
df_transcript['aa_length'] = df_transcript['aa_length'].astype('Int64')

In [ ]:
# [extra] Normalize missing values
import numpy as np
df_transcript = df_transcript.replace([r'^\s*$', 'None', 'none', 'NA', 'na', 'null', 'Null'], np.nan, regex=True)

In [ ]:
# Generate amino-acid information consistently
from Bio.Seq import Seq
import warnings
warnings.filterwarnings("ignore")


def translate_cds(cds):
	if pd.isna(cds):
		return cds
	try:
		seq = Seq(cds)
		amino_acid_seq = seq.translate(to_stop=True)
		return str(amino_acid_seq).upper()
	except:
		return None

		
df_transcript['aa_sequence'] = df_transcript['coding_sequence'].apply(translate_cds) # Translate the sequence
df_transcript['aa_length'] = df_transcript['aa_sequence'].apply(lambda x: len(x) if isinstance(x, str) else None) # Amino-acid sequence length

In [ ]:
df_transcript['aa_length'] = df_transcript['aa_length'].astype('Int64')
df_transcript['strand'] = df_transcript['strand'].astype('Int64')
df_transcript.to_csv(transcript_path, sep='\t', index=False)

### (E) Refine the retained data using transcript information

- Step (C) initially filtered samples by transcript identifiers, and step (D) further retained transcripts with coding information. This step applies the second filter.

In [ ]:
df_transcript_refseq = df_transcript[df_transcript['refseq_id'].notna()]
print(f"[INFO] Available RefSeq transcript records: {len(df_transcript_refseq)}")
df_transcript_refseq.dropna(subset=['strand', 'coding_sequence', 'aa_sequence'], inplace=True)
print(f"[INFO] Available RefSeq transcript records(including annotations): {len(df_transcript_refseq)}")

In [ ]:
# Apply a second filter using transcript information
print(f"[INFO] Sanity check 1 | Pathogenic variants: {len(df_mis_pt)} | Benign variants: {len(df_mis_bn)}")
df_mis_pt = df_mis_pt[df_mis_pt['refseq'].isin(df_transcript_refseq['refseq_id'])]
df_mis_bn = df_mis_bn[df_mis_bn['refseq'].isin(df_transcript_refseq['refseq_id'])]
print(f"[INFO] Valid transcripts | Pathogenic variants: {len(df_mis_pt)} | Benign variants: {len(df_mis_bn)}")

print(f"[INFO] Sanity check 1: {len(df_pairs)}")
df_pairs = df_pairs[(df_pairs['neg_variant38'].isin(df_mis_bn['variant38'])) & (df_pairs['pos_variant38'].isin(df_mis_pt['variant38']))]
df_pairs.to_csv(f"../dataset/a-middleware/missense/PAIRS_missnese.sanity.check.2.txt", sep='\t', index=False)
print(f"[INFO] After remove non-info records: {len(df_pairs)}")

df_mis_bn = df_mis_bn[df_mis_bn['variant38'].isin(df_pairs['neg_variant38'])]
df_mis_bn.to_csv(f"../dataset/b-output/missense_dataset/benign.with_transcript.sanity.check.2.txt", sep='\t', index=False)
df_mis_pt = df_mis_pt[df_mis_pt['variant38'].isin(df_pairs['pos_variant38'])]
df_mis_pt.to_csv(f"../dataset/b-output/missense_dataset/pathogenic.with_transcript.sanity.check.2.txt", sep='\t', index=False)

# 3 Source-domain dataset overview

## 3.1 Assign mutype labels to non-coding variants

HGMD and ClinVar use different mutype descriptions. ClinVar uses VEP consequences, whereas HGMD provides only canonical-splice, splice, and regulatory categories. Regulatory variants are annotated with VEP here to obtain specific consequences.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")


def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
	variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
	return variant

### (A) ClinVar labels

In [ ]:
# Positive non-coding variants
noncoding_pos_df = pd.read_csv("../dataset/b-output/noncoding_dataset/noncoding.pt.cleaned.txt", sep='\t')
noncoding_pos_df['label'] = [1] * len(noncoding_pos_df)
noncoding_pos_df['variant38'] = retrieve_variant(noncoding_pos_df, '#CHROM', 'POS', 'REF', 'ALT')
print(f"[INFO] amount of pos noncoding: {len(noncoding_pos_df)}")

# Obtain ClinVar labels
clinvar_pos_df = pd.read_csv("../dataset/a-middleware/ClinVar.noncoding.pathogenic.txt", sep='\t', usecols=['variant38', 'CLNSIG', 'MC_filtered'])
clinvar_pos_df['source'] = ['clinvar'] * len(clinvar_pos_df) 
noncoding_pos_df = noncoding_pos_df.merge(clinvar_pos_df, how='left', left_on='variant38', right_on='variant38', suffixes=('', '_m'))
print(f"[INFO] amount of pos noncoding (after merged): {len(noncoding_pos_df)}")
noncoding_pos_df.rename(columns={'CLNSIG': 'evidence_level', 'MC_filtered': 'mutype'}, inplace=True)
noncoding_pos_df.head()

### (B) HGMD labels
HGMD divides non-coding consequences into canonical splice, splice, and regulatory categories.
The regulatory category is not further subdivided.

In [ ]:
# Obtain HGMD labels
hgmd_df = pd.read_csv("../dataset/a-middleware/HGMD.noncodingSNV.dm+dmq.txt", sep='\t', usecols=['variant_hg38', 'mutype', 'tag'])
hgmd_df.rename(columns={'tag': 'evidence_level', 'variant_hg38': 'variant38'}, inplace=True)
hgmd_df['source'] = ['hgmd'] * len(hgmd_df)

# For the source, evidence_level, and mutype columns 
# Append HGMD information when an original value exists (for example, `original,hgmd`); otherwise use the hgmd_df value.
noncoding_pos_df_merged = noncoding_pos_df.merge(hgmd_df, how='left', on='variant38', suffixes=('', '_m'))

def combine_values(orig, new):
	"""General helper for information columns"""
	if pd.isna(new):  # HGMD provides no new value
		return orig
	if pd.isna(orig):  # Use the new value when the original is NaN
		return new
	return f"{orig},{new}"  # Concatenate when both values exist

# Process the three columns separately
for col in ['source', 'evidence_level', 'mutype']:
	noncoding_pos_df_merged[col] = [
		combine_values(o, n)
		for o, n in zip(noncoding_pos_df_merged[col], noncoding_pos_df_merged[f'{col}_m'])
	]

# Remove redundant *_m columns
noncoding_pos_df_merged.drop(columns=[f'{c}_m' for c in ['source', 'evidence_level', 'mutype']], inplace=True)
print(f"[INFO] After merged: {len(noncoding_pos_df_merged)}")
noncoding_pos_df_merged.head()

### (C) Annotate regulatory variants

In [ ]:
def reformat_as_vcf(input_vcf):
	"""Convert the VCF table to standard VCF format"""
	input_vcf['ID'] = ['.'] * len(input_vcf)
	input_vcf['FILTER'] = ['.'] * len(input_vcf)
	input_vcf['INFO'] = ['.'] * len(input_vcf)
	input_vcf['QUAL'] = ['.'] * len(input_vcf)
	input_vcf = input_vcf[['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']]
	# Convert the chromosome column to string type
	input_vcf['#CHROM'] = input_vcf['#CHROM'].astype(str)
	# #CHROM sort as 1-22, X, Y and then by ascending POS within each chromosome
	def chrom_order(chrom):
		# Remove an optional chr prefix
		c = chrom.replace('chr', '')
		# Sort numbered chromosomes as integers
		if c.isdigit():
			return int(c)
		# X, Y, M Handle separately
		elif c == 'X':
			return 23
		elif c == 'Y':
			return 24
		elif c in ['M', 'MT']:
			return 25
		else:
			return 26  # Place other unknown chromosomes last
	input_vcf = input_vcf.sort_values(
		by=['#CHROM', 'POS'],
		key=lambda col: col.map(chrom_order) if col.name == '#CHROM' else col
	).reset_index(drop=True)
	return input_vcf


# Annotate HGMD regulatory variants with specific consequences
nc_hgmd_regu = noncoding_pos_df_merged[noncoding_pos_df_merged['mutype'] == 'regulatory']
nc_hgmd_regu.to_csv("../dataset/a-middleware/noncoding/pathogenic.regulatory.txt", sep='\t', index=False)
nc_hgmd_regu_format = reformat_as_vcf(nc_hgmd_regu)
nc_hgmd_regu_format.to_csv("../dataset/a-middleware/noncoding/pathogenic.regulatory.format.txt", sep='\t', index=False)

In [ ]:
# Annotate regulatory variants for which HGMD provides no specific functional consequence
# VEP was installed on the original analysis server
conda activate vep
cd ../VEPandPlugins/
./RunVep.sh # Set the script input and output paths

In [ ]:
nc_hgmd_regu_format_output = pd.read_csv("../dataset/a-middleware/noncoding/pathogenic.regulatory.format.vep.output.txt", sep='\t', comment='#', header=None, usecols=[0, 6])
nc_hgmd_regu_format_output.columns = ['variant38', 'consequence']
# For VEP results, use the most severe non-coding consequence as the final result (these are known regulatory variants, so coding consequences are ignored). Consequence ranks follow VEP: https://asia.ensembl.org/info/genome/variation/prediction/predicted_data.html?
dt = {
	'splice_acceptor_variant': 4, # Coding
	'splice_donor_variant': 4, # Coding
	'stop_lost': 4, # Coding
	'stop_gained': 4, # Coding
	'initiator_codon_variant': 4, # Coding
	'nonsense': 4, # Coding
	'start_lost': 4, # Coding
	'missense_variant': 3, # Coding
	'splice_region_variant': 2, # NC
	'splice_donor_region_variant': 2, # NC
	'splice_polypyrimidine_tract_variant': 2, # NC
	'splice_donor_5th_base_variant': 2, # NC
	'synonymous_variant': 2, # Coding
	'incomplete_terminal_codon_variant': 2, # Coding
	'stop_retained_variant': 2, # Coding
	'coding_sequence_variant': 1, # Coding
	'mature_miRNA_variant': 1,  # NC
	'non_coding_transcript_variant': 1, # NC
	'5_prime_UTR_variant': 1, # NC
	'3_prime_UTR_variant': 1, # NC
	'intron_variant': 1, # NC
	'genic_downstream_transcript_variant': 1, # NC
	'genic_upstream_transcript_variant': 1, # NC
	'upstream_gene_variant': 1, # NC
	'downstream_gene_variant': 1, # NC
	'NMD_transcript_variant': 1, # NC
	'non_coding_transcript_exon_variant': 1, # NC
	'non_coding_transcript_variant': 1, # NC
	'intergenic_variant': 1, # NC
	'no_sequence_alteration': 0, # No sequence alteration
}

# NOTE: Target set (retained non-coding variant types only). ClinVar MC terms and VEP consequences differ for categories such as upstream/downstream and non-coding transcript variants; distinguish them carefully.
# NOTE: In HGMD, regulatory variants are distinct from splice/canonical_splice variants; therefore splice_acceptor_variant and splice_donor_variant are excluded here.
target_set = {
	'splice_region_variant',
	'splice_donor_region_variant',
	'splice_polypyrimidine_tract_variant',
	'splice_donor_5th_base_variant',
	'mature_miRNA_variant',
	'non_coding_transcript_variant',
	'5_prime_UTR_variant',
	'3_prime_UTR_variant',
	'intron_variant',
	'upstream_gene_variant',
	'downstream_gene_variant',
	'NMD_transcript_variant',
	'non_coding_transcript_exon_variant',
	'non_coding_transcript_variant',
	'intergenic_variant'
}

def process_consequences(group):
	"""Process all consequence records for each variant38"""
	all_consequences = set()
	# Iterate over all records for variant38, then split and filter consequences
	for conseq_str in group['consequence']:
		if pd.isna(conseq_str):
			continue
		# Split comma-separated consequences
		conseq_list = [c.strip() for c in conseq_str.split(',')]
		# Retain only types in target_set
		filtered_conseq = [c for c in conseq_list if c in target_set]
		all_consequences.update(filtered_conseq)
	
	if not all_consequences:
		return None
	
	# Calculate the score for each consequence
	conseq_scores = {}
	for conseq in all_consequences:
		score = dt.get(conseq, 0)  # Default to zero when absent from dt
		if score not in conseq_scores:
			conseq_scores[score] = []
		conseq_scores[score].append(conseq)
	
	# Find the maximum score
	max_score = max(conseq_scores.keys())
	max_consequences = conseq_scores[max_score]
	
	# Join tied consequences with commas; otherwise return the single result
	if len(max_consequences) > 1:
		return ','.join(sorted(max_consequences))  # Sort for deterministic results
	else:
		return max_consequences[0]

# Process groups defined by variant38
processed_data = []
for variant38, group in nc_hgmd_regu_format_output.groupby('variant38'):
	result = process_consequences(group)
	if result is not None:
		processed_data.append({'variant38': variant38, 'consequence': result})

nc_hgmd_regu_processed = pd.DataFrame(processed_data)
print("Records before processing:", len(nc_hgmd_regu_format))
print("Number of VEP annotations:", len(nc_hgmd_regu_format_output))
print("Unique variant38 values after processing:", len(nc_hgmd_regu_processed))
print("\nPreview of processed data:")
nc_hgmd_regu_processed.head()

In [ ]:
# Variant types excluded after VEP annotation: all VEP consequences for the following four variants are coding consequences, so they are excluded from the totals
undistinguished_set = set(nc_hgmd_regu[~nc_hgmd_regu['variant38'].isin(nc_hgmd_regu_processed['variant38'])]['variant38'])
nc_hgmd_regu_format_output[nc_hgmd_regu_format_output['variant38'].isin(undistinguished_set)]

### (D) Merge all variant types

In [ ]:
import numpy as np

nc_hgmd_regu_processed.rename(columns={'consequence': 'mutype'}, inplace=True)

noncoding_pos_df_merged = noncoding_pos_df_merged.merge(
	nc_hgmd_regu_processed, how='left', on='variant38', suffixes=('', '_m')
)

# Replace the original mutype with mutype_m only when a match exists
noncoding_pos_df_merged['mutype'] = np.where(
	noncoding_pos_df_merged['mutype_m'].notna(),
	noncoding_pos_df_merged['mutype_m'],
	noncoding_pos_df_merged['mutype']
)

# Remove redundant columns
noncoding_pos_df_merged.drop(columns=['mutype_m'], inplace=True)
print("[INFO] Final number of processed records:", len(noncoding_pos_df_merged))
noncoding_pos_df_merged.head()

In [ ]:
# Save the final data
noncoding_pos_df_merged.to_csv("../dataset/a-middleware/noncoding/pathogenic.all.mutype.source.evidence.txt", sep='\t', index=False)

### (E) Benign variant types

In [ ]:
noncoding_bn = pd.read_csv("../dataset/b-output/noncoding_dataset/noncoding.bn.cleaned.txt", sep='\t')
noncoding_bn['variant38'] = retrieve_variant(noncoding_bn, '#CHROM', 'POS', 'REF', 'ALT')
noncoding_bn.drop(['#CHROM', 'POS', 'REF', 'ALT'], axis=1, inplace=True)
clinvar_bn = pd.read_csv("../dataset/a-middleware/ClinVar.noncoding.benign.txt", sep='\t', usecols=['variant38', 'MC_filtered'])
noncoding_bn = noncoding_bn.merge(clinvar_bn, how='left', left_on='variant38', right_on='variant38', suffixes=('', '_m'))
noncoding_bn.head()

In [ ]:
noncoding_bn.rename(columns={'MC_filtered': 'mutype'}, inplace=True)

## 3.2 Summarize the internal dataset

### (A) 🚩 Sample-source distribution

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_mut_source(df, title, savepath=None):
	# 1 Count the source distribution
	counts = df['source'].value_counts()
	# 2 Define label mappings
	label_map = {
		'clinvar': 'ClinVar',
		'hgmd': 'HGMD',
		'clinvar,hgmd': 'ClinVar + HGMD'
	}
	# 3 Define color mappings
	color_map = {
		'clinvar': '#cacaca',
		'hgmd': '#f8eedd',
		'clinvar,hgmd': '#c82524'
	}
	# 4 Extract labels and colors
	labels = [label_map.get(src, src) for src in counts.index]
	colors = [color_map.get(src, '#dddddd') for src in counts.index]
	# 5 Draw a donut chart
	fig, ax = plt.subplots(figsize=(5, 5))
	wedges, texts, autotexts = ax.pie(
		counts,
		colors=colors,
		startangle=90,
		wedgeprops=dict(width=0.4, edgecolor='w'),  # Width controls the donut hole
		autopct='%1.1f%%',
		pctdistance=0.8,  # Move percentage labels outward
		textprops={'color': 'black', 'fontsize': 18}  # Increase font size
	)
	# 6 Draw the legend
	ax.legend(wedges, labels,
			frameon=False,
			loc="lower center",
			fontsize=16,
			ncols=3,
			bbox_to_anchor=(0.25, -0.12, 0.5, 0.5),  # Adjust legend position
			handlelength=1,  # Set marker length
			handleheight=1,  # Set marker height
			handletextpad=0.5,  # Set spacing between markers and text
			columnspacing=0.4,
			labelspacing=0.3)  # Set boundary padding
	# 7 Set the title and equal aspect ratio
	ax.set_title(title, fontsize=18)
	ax.axis('equal')
	if savepath:
		# Save with adjusted boundaries for a compact figure
		plt.savefig(savepath, dpi=300, bbox_inches='tight')
	plt.show()

In [ ]:
# Positive non-coding variants
plot_mut_source(df=noncoding_pos_df_merged,
				title="Source distribution (positive noncoding variants)",
				savepath="../dataset/c-statistic/Noncoding.source.jpg")

In [ ]:
# Positive missense variants
pt_expand = pd.read_csv("../dataset/b-output/missense_dataset/pathogenic.rmSyn.txt", sep='\t')
missense_all_pt = pd.read_csv("../dataset/a-middleware/missense/pathogenic.txt", sep='\t', usecols=['variant38', 'source'])
print(f"[INFO] Before merge: {len(pt_expand)}")
pt_expand = pt_expand.merge(missense_all_pt, how='left', left_on='variant38', right_on='variant38', suffixes=('', '_m'))
print(f"[INFO] After merge: {len(pt_expand)}")
pt_expand['source'] = pt_expand['source'].str.replace(';', ',')
pt_expand.head()

In [ ]:
plot_mut_source(df=pt_expand,
				title="Source distribution (positive missense variants)",
				savepath="../dataset/c-statistic/missense.source.jpg")

### (B) 🚩 Non-coding mutype distribution
Variant hierarchy reference:  https://www.ebi.ac.uk/ols4/search?q=3_prime_UTR_variant&ontology=so

| Primary category | Subtype | Description |
|---|---|---|
| **Intergenic** | `intergenic_variant`, `upstream_gene_variant`, `genic_upstream_transcript_variant`, `downstream_gene_variant` | Outside genes or in upstream/downstream regulatory regions |
| **UTR variants** | `5_prime_UTR_variant`, `3_prime_UTR_variant` | In mRNA untranslated regions, potentially affecting translation efficiency or stability |
| **Intronic variants**  | `intron_variant` | Within introns  |
| **Non-coding transcript variants** | `non_coding_transcript_variant`, `non_coding_transcript_exon_variant`, `mature_miRNA_variant`, `NMD_transcript_variant` | Affects non-coding RNA or nonsense-mediated decay |
| **Splice-related variants** | **Canonical:** `canonical_splice` (donor and acceptor combined)<br>**Non-canonical:** `splice`, `splice_donor_5th_base_variant`, `splice_donor_region_variant`, `splice_polypyrimidine_tract_variant`, `splice_region_variant` | Splice-related variants, divided into core and extended splice sites |


- Noncoding Variant
	- Intergeni
		- intergenic_variant
		- upstream_gene_variant
		- genic_upstream_transcript_variant
		- downstream_gene_variant
	- UTR Variant
		- 5_prime_UTR_variant
		- 3_prime_UTR_variant
	- Intronic Variant
		- intron_variant
	- Non-coding Transcript Variant
		- non_coding_transcript_variant
		- non_coding_transcript_exon_variant
		- mature_miRNA_variant
		- NMD_transcript_variant
	- Splicing Variant
		- Canonical: 
			- splice_donor_variant
			- splice_acceptor_variant 
		- Non-canonical:
			- splice
			- splice_donor_5th_base_variant
			- splice_donor_region_variant
			- splice_polypyrimidine_tract_variant
			- splice_region_variant


#### Normalization

In [ ]:
def standardize_mutype(mutype_str):
	"""Standardize the variant-type column"""
	if pd.isna(mutype_str):
		return mutype_str
	
	# Step 1: remove unwanted types
	types_to_remove = ['regulatory', 'nonsense', 'stop_lost', 'initiator_codon_variant']
	
	# Split and filter
	mutypes = mutype_str.split(',')
	filtered_mutypes = [mutype for mutype in mutypes if mutype not in types_to_remove]
	
	# Return an empty string if no types remain
	if not filtered_mutypes:
		return ''
	
	# Step 2: standardize names
	standardized_mutypes = []
	for mutype in filtered_mutypes:
		if mutype == 'canonical-splice':
			standardized_mutypes.append('canonical_splice')
		elif mutype == 'non-coding_transcript_variant':
			standardized_mutypes.append('non_coding_transcript_variant')
		else:
			standardized_mutypes.append(mutype)
	
	# Return standardized values joined by commas
	return ','.join(standardized_mutypes)

# positiveApply the standardization function to create a new column
noncoding_pos_df_merged['format_mutype'] = noncoding_pos_df_merged['mutype'].apply(standardize_mutype)
noncoding_pos_df_merged['format_mutype'].value_counts()

In [ ]:
# negative Apply the standardization function to create a new column
noncoding_bn['format_mutype'] = noncoding_bn['mutype'].apply(standardize_mutype)
noncoding_bn['format_mutype'].value_counts()

#### Calculate category mappings

In [ ]:
# Step 1. Expand variant types
def explode_mutypes(df, col='format_mutype'):
	tmp = df[col].dropna().str.split(',')
	return pd.DataFrame({'mutype': [m.strip() for sublist in tmp for m in sublist if m]})

# Step 2. Map primary categories
def classify_mutype(mutype):
	if mutype in ['intergenic_variant', 'upstream_gene_variant', 'genic_downstream_transcript_variant',
				  'genic_upstream_transcript_variant', 'downstream_gene_variant']:
		return 'Intergenic'
	elif mutype in ['5_prime_UTR_variant', '3_prime_UTR_variant']:
		return 'UTR'
	elif mutype in ['intron_variant']:
		return 'Intronic'
	elif mutype in ['non_coding_transcript_variant',
					'non_coding_transcript_exon_variant',
					'mature_miRNA_variant', 'NMD_transcript_variant']:
		return 'Non-coding transcript'
	elif mutype in ['canonical_splice', 'splice', 'splice_acceptor_variant',
					'splice_donor_5th_base_variant', 'splice_donor_region_variant',
					'splice_donor_variant', 'splice_polypyrimidine_tract_variant',
					'splice_region_variant']:
		return 'Splicing'
	else:
		return 'Other'

# Step 3. Subtype
def map_subtype(row):
	if row['major_class'] == 'Splicing':
		if row['mutype'] in ['canonical_splice', 'splice_acceptor_variant', 'splice_donor_variant']:
			return 'Canonical splice site variants'
		else:
			return 'Non-canonical splice site variants'
	else:
		# Use the original name for other types
		return row['mutype']

# Step 4. Replace underscores with spaces
def format_and_count(df, save_path):
	pos_mutype_df = explode_mutypes(df)
	pos_mutype_df['major_class'] = pos_mutype_df['mutype'].apply(classify_mutype)
	pos_mutype_df['subtype'] = pos_mutype_df.apply(map_subtype, axis=1)
	# Step 4. Replace underscores with spaces
	pos_mutype_df['subtype'] = pos_mutype_df['subtype'].str.replace('_', ' ')
	pos_mutype_df['major_class'] = pos_mutype_df['major_class'].str.replace('_', ' ')
	# Step 5. Count records
	pos_count_df = pos_mutype_df.groupby(['major_class', 'subtype']).size().reset_index(name='count')
	pos_count_df.to_csv(save_path, index=False)
	return pos_count_df

In [ ]:
# Positive variants
pos_count_df = format_and_count(df=noncoding_pos_df_merged, 
								save_path="../dataset/c-statistic/noncoding.postive.mutype.csv")
pos_count_df

In [ ]:
# Negative variants
neg_count_df = format_and_count(df=noncoding_bn, 
								save_path="../dataset/c-statistic/noncoding.negative.mutype.csv")
neg_count_df

### (C) 🚩 Variant-category visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_circle_matplotlib(count_df, save_path=None):
	# ---- Aggregate data ----
	major_counts = count_df.groupby('major_class')['count'].sum()
	sub_counts = count_df.groupby(['major_class', 'subtype'])['count'].sum()

	parent_colors = {
		'Splicing': '#15607a',
		'UTR': '#c71e1d',
		'Intergenic': '#ff8902',
		'Non-coding transcript': '#019176',
		'Intronic': '#c5d8e5'
	}

	child_colors = {
		'Canonical splice site variants': '#19a1cd',
		'Non-canonical splice site variants': '#1d81a2',
		'5 prime UTR variant': '#efb1a5',
		'3 prime UTR variant': '#ff483b',
		'intergenic variant': '#ff8902',
		'upstream gene variant': '#ffac63',
		'genic upstream transcript variant': '#ffce83',
		'genic downstream transcript variant': '#ffdbc1',
		'downstream gene variant': '#ffcea9',
		'non coding transcript variant': '#0abb9f',
		'non coding transcript exon variant': '#01dca6',
		'mature miRNA variant': '#58f8b4',
		'NMD transcript variant': '#8af4c1',
		'intron variant': '#c5d8e5'
	}

	outer_labels = major_counts.index.tolist()
	outer_vals = major_counts.values
	outer_colors = [parent_colors[k] for k in outer_labels]

	inner_labels = sub_counts.index.get_level_values(1).tolist()
	inner_vals = sub_counts.values
	inner_colors = [child_colors[k] for k in inner_labels]

	# ---- Plot ----
	fig, ax = plt.subplots(figsize=(9, 8), dpi=150)
	size = 0.3

	# Outer ring: show labels and percentages
	wedges1, texts1, autotexts1 = ax.pie(
		outer_vals, radius=1, colors=outer_colors,
		wedgeprops=dict(width=size, edgecolor='w'),
		labels=outer_labels, labeldistance=1.08,
		autopct='%1.1f%%', pctdistance=0.85,
		textprops={'fontsize': 14, 'weight': 'bold'}
	)

	# Inner ring: hide labels and percentages
	wedges2, _ = ax.pie(
		inner_vals, radius=1-size, colors=inner_colors,
		wedgeprops=dict(width=size, edgecolor='w'),
		labels=None,  # Hide labels
		autopct=None  # Hide percentages
	)

	# Set the title and style
	# ax.set(aspect="equal", title='Consequences (negative noncoding variants)')
	# plt.title("Consequences (negative noncoding variants)", fontsize=18,  y=0.92)
	plt.title("Consequences (positive noncoding variants)", fontsize=18,  y=0.92)

	# Use the legend for the inner ring only
	plt.legend(
		wedges2, inner_labels,
		bbox_to_anchor=(0.55, -0.1),
		loc="lower center",
		fontsize=14,
		ncols=2,
		frameon=True,
		handlelength=1,  # Set marker length
		handleheight=1,  # Set marker height
		handletextpad=0.5,  # Set spacing between markers and text
		columnspacing=0.4,
	)

	plt.tight_layout()
	if save_path:
		plt.savefig(save_path, bbox_inches='tight')
		print(f"[INFO] Saved: {save_path}")


In [ ]:
# positive noncoding variants
# plot_circle_matplotlib(count_df=pos_count_df, save_path=None) 
plot_circle_matplotlib(count_df=pos_count_df, save_path="../dataset/c-statistic/noncoding_variant_sunburst_positive.png") 

In [ ]:
# negative noncoding variants
plot_circle_matplotlib(count_df=neg_count_df, save_path="../dataset/c-statistic/noncoding_variant_sunburst_negative.png") 

### (D) 🚩 Evidence levels

Retain original subcategories within each primary category

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def count_by_evidence(df):
	mapping = {
		"DM?": "Single source, likely/questionable",
		"Likely_pathogenic": "Single source, likely/questionable",
		"Likely_pathogenic,DM?": "Dual sources, likely/questionable",
		"DM": "Single source, confirmed",
		"Pathogenic": "Single source, confirmed",
		"Pathogenic/Likely_pathogenic": "Single source, confirmed",
		"Likely_pathogenic,DM": "Dual sources, one source confirmed",
		"Pathogenic/Likely_pathogenic,DM": "Dual sources, confirmed",
		"Pathogenic,DM?": "Dual sources, one source confirmed",
		"Pathogenic/Likely_pathogenic,DM?": "Dual sources, one source confirmed",
		"Pathogenic,DM": "Dual sources, confirmed"
	}
	stat_df = pd.DataFrame({
		'subtype': list(df['evidence_level'].value_counts().keys()),
		'count': list(df['evidence_level'].value_counts())
	})
	stat_df['major_class'] = stat_df['subtype'].map(mapping)
	return stat_df

def plot_evidence_level_donut(count_df, title, savepath=None):
	# Aggregate data
	major_counts = count_df.groupby('major_class')['count'].sum()
	sub_counts = count_df.groupby(['major_class', 'subtype'])['count'].sum()
	# Define colors
	parent_colors = {
		"Single source, likely/questionable": "#f9e0a3", # #f9b6a6
		"Dual sources, likely/questionable": "#f8d182", # #f09581
		"Single source, confirmed": "#fac062", # #e5735e
		"Dual sources, one source confirmed": "#fcaf43", # #d74f3d
		"Dual sources, confirmed": "#ff9b21" # #c71e1d
	}
	child_colors = {
		"DM?": '#a082be',
		"Likely_pathogenic": '#d9cde5',
		"Likely_pathogenic,DM?": '#4e416b',
		"DM": '#b6b1c4',
		"Pathogenic": '#75406f',
		"Pathogenic/Likely_pathogenic": '#c7b2bf',
		"Likely_pathogenic,DM": '#796eb0',
		"Pathogenic/Likely_pathogenic,DM": '#c9c4e5',
		"Pathogenic,DM?": '#62598f',
		"Pathogenic/Likely_pathogenic,DM?": '#c0bdce',
		"Pathogenic,DM": '#000000'
	}
	outer_labels = major_counts.index.tolist()
	outer_vals = major_counts.values
	outer_colors = [parent_colors[k] for k in outer_labels]

	inner_labels = sub_counts.index.get_level_values(1).tolist()
	inner_vals = sub_counts.values
	inner_colors = [child_colors[k] for k in inner_labels]

	# Plot
	fig, ax = plt.subplots(figsize=(9, 12), dpi=150)
	size = 0.3

	# Outer ring: show labels and percentages
	wedges1, texts1, autotexts1 = ax.pie(
		outer_vals, radius=1, colors=outer_colors,
		wedgeprops=dict(width=size, edgecolor='w'),
		labels=outer_labels, labeldistance=1.08,
		autopct='%1.1f%%', pctdistance=0.85,
		textprops={'fontsize': 15}
	)

	# Inner ring: hide labels and percentages
	wedges2, _ = ax.pie(
		inner_vals, radius=1-size, colors=inner_colors,
		wedgeprops=dict(width=size, edgecolor='w'),
		labels=None,  # Hide labels
		autopct=None  # Hide percentages
	)

	# Set the title and style
	# ax.set(aspect="equal", title='Consequences (negative noncoding variants)')
	# plt.title("Consequences (negative noncoding variants)", fontsize=18,  y=0.92)
	# plt.title(title, fontsize=18,  y=0.95)
	plt.title(title, fontsize=18, x=0.7, y=0.95)

	# Use the legend for the inner ring only
	plt.legend(
		wedges2, inner_labels,
		# bbox_to_anchor=(0.55, -0.4),
		bbox_to_anchor=(0.65, -0.4),
		loc="lower center",
		fontsize=14,
		ncols=2,
		frameon=True,
		edgecolor='black',
		handlelength=1,  # Set marker length
		handleheight=1,  # Set marker height
		handletextpad=0.5,  # Set spacing between markers and text
		columnspacing=0.4,
	)

	plt.tight_layout()
	if savepath:
		plt.savefig(savepath, bbox_inches='tight')
		print(f"[INFO] Saved: {savepath}")

#### 🔹 Missense variants

In [ ]:
pt_expand = pd.read_csv("../dataset/b-output/missense_dataset/missense.pt.rmSyn.with.detailed.txt", sep='\t')
pt_expand_stat = count_by_evidence(pt_expand)

In [ ]:
plot_evidence_level_donut(count_df=pt_expand_stat,
						  title="Evidence levels (positive missense variants)")

#### 🔹 Non-coding variants

In [ ]:
noncoding_pos_df_merged = pd.read_csv("../dataset/a-middleware/noncoding/pathogenic.all.mutype.source.evidence.txt", sep='\t')

In [ ]:
noncoding_pos_df_merged_stat = count_by_evidence(noncoding_pos_df_merged)
plot_evidence_level_donut(count_df=noncoding_pos_df_merged_stat,
						  title="Evidence levels (positive noncoding variants)")

## 3.3 🚩 Variant overlap and non-overlapping variants

1. Both non-coding and splice datasets use close-by pairing.
2. In the non-coding dataset, close-by pairing yields more intronic negative samples.
3. In the splice dataset, close-by pairing yields more negative splice variants than positive variants, so not all positive samples are used.

In [ ]:
import pandas as pd


NONCODING_PATH = "../dataset/b-output/noncoding_dataset/"
df_noncoding_bn = pd.read_csv(f"{NONCODING_PATH}/noncoding.bn.cleaned.txt", sep='\t')
df_noncoding_bn['variant38'] = retrieve_variant(df_noncoding_bn, '#CHROM', 'POS', 'REF', 'ALT')
df_noncoding_pt = pd.read_csv(f"{NONCODING_PATH}/noncoding.pt.cleaned.txt", sep='\t')
df_noncoding_pt['variant38'] = retrieve_variant(df_noncoding_pt, '#CHROM', 'POS', 'REF', 'ALT')
df_noncoding = pd.concat([df_noncoding_bn, df_noncoding_pt], ignore_index=True)

MISSENSE_PATH = "../dataset/b-output/missense_dataset/"
df_missense_bn = pd.read_csv(f"{MISSENSE_PATH}/missense.bn.rmSyn.txt", sep='\t')
df_missense_pt = pd.read_csv(f"{MISSENSE_PATH}/missense.pt.rmSyn.txt", sep='\t')
df_missense = pd.concat([df_missense_bn, df_missense_pt], ignore_index=True)

In [ ]:
from matplotlib_venn import venn2
import matplotlib.pyplot as plt

def draw_venn(set1, set2, set1_name, set2_name, title_name, color_set, save_path=None):
	fig, ax = plt.subplots(figsize=(5, 5))
	venn = venn2(
		[set1, set2],
		set_labels=(set1_name, set2_name),
		set_colors=color_set,
		alpha=0.8,
		ax=ax
	)
	# Adjust fonts
	for text in venn.set_labels:
		text.set_fontsize(16)
	for text in venn.subset_labels:
		if text:
			# Format numbers with thousands separators
			try:
				num = int(text.get_text())
				text.set_text(f"{num:,}")  # Thousands separator
			except ValueError:
				pass
			text.set_fontsize(16)

	ax.set_aspect('equal')
	plt.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)
	plt.title(title_name, fontsize=18, y=0.95)
	if save_path:
		plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.1)
	plt.show()

# 4 Figure 1(B) - Overview of pathogenic variant counts

- Compare pathogenic variant counts in ClinVar version 2025.04
- Display as a pie chart


cd ../dataset/a-middleware/

```python
>>> import pandas as pd
>>> df = pd.read_csv("ClinVar.reviewed.txt", sep='\t')
>>> df['CLNSIG'].value_counts()
```

Pathogenic                                                   63257
Likely_pathogenic                                            60634
Pathogenic/Likely_pathogenic                                 19805
...

```python
>>> df_p = df[(df['CLNSIG'] == 'Pathogenic') | (df['CLNSIG'] == 'Likely_pathogenic') | (df['CLNSIG'] == 'Pathogenic/Likely_pathogenic')]
>>> df_p['MC'].value_counts()
```

SO:0001587|nonsense 45723
SO:0001583|missense_variant 34659
SO:0001575|splice_donor_variant 16808
...

## 4.1 Extract the number of pathogenic nonsense variants

In [ ]:
# clinvar
import pandas as pd

df = pd.read_csv("../dataset/a-middleware/ClinVar.reviewed.txt", sep='\t', usecols=['variant38', 'MC', 'CLNSIG'])
df_p = df[(df['CLNSIG'] == 'Pathogenic') | (df['CLNSIG'] == 'Likely_pathogenic') | (df['CLNSIG'] == 'Pathogenic/Likely_pathogenic')]
df_nonsense = df_p[df_p['MC'] == 'SO:0001587|nonsense']
print(len(df_nonsense))
df_nonsense.to_csv("../dataset/a-middleware/ClinVar.nonsense.pathogenic.txt", sep='\t', index=False)

In [ ]:
# hgmd
import pandas as pd
import warnings
warnings.filterwarnings("ignore") # Suppress all warnings


def retrieve_variant(df, chrom_col, pos_col, ref_col, alt_col, prefix=''):
	variant = [f'{prefix}{chro}_{pos}_{ref}/{alt}' for chro, pos, ref, alt in zip(df[chrom_col], df[pos_col], df[ref_col], df[alt_col])]
	return variant


hgmd_usecols = ['acc_num', 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38', 'amino', 'disease', 'tag', 'mutype', 'refseq', 'hgvs', 'codon']
df = pd.read_csv("../dataset/x-source-data/HGMD_2023.3_allmut.csv", usecols=hgmd_usecols)
df['pos_hg38'] = df['pos_hg38'].astype('Int64') # Nullable integer type
df['chrom_hg38'] = df['chrom_hg38'].astype(str) 
nucleotides = {'A', 'C', 'T', 'G'}
chr_list = [str(i) for i in range(1, 23)] + ['X', 'Y']

nonsense_df = df[
	(df['chrom_hg38'].isin(chr_list)) &
	(df['mutype'] == 'nonsense') &
	(df['pos_hg38'].notna()) &
	(df['ref_hg38'].isin(nucleotides)) &
	(df['alt_hg38'].isin(nucleotides)) &
	(df['ref_hg38'] != df['alt_hg38'])
]
print(f'[INFO] Number of nonsense variants:: {len(nonsense_df)}')

nonsense_dm_plus_df = nonsense_df[(nonsense_df['tag'] == 'DM') | (nonsense_df['tag'] == 'DM?')]
print(f'[INFO] Number classified as DM or DM?:: {len(nonsense_dm_plus_df)}')

nonsense_dm_plus_df['variant_hg38'] = retrieve_variant(nonsense_dm_plus_df, 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38')
nonsense_dm_plus_df.drop_duplicates(subset='variant_hg38', inplace=True)
print(f"[INFO] Remaining after deduplication by chromosome, position, reference allele, and alternate allele:: {len(nonsense_dm_plus_df)}  records")
nonsense_dm_plus_df.to_csv("../dataset/a-middleware/HGMD.nonsense.dm+dmq.txt", sep='\t', index=False)
nonsense_dm_df = nonsense_dm_plus_df[(nonsense_dm_plus_df['tag'] == 'DM')]
nonsense_dm_df.to_csv("../dataset/a-middleware/HGMD.nonsense.dm.txt", sep='\t', index=False)

## 4.2 Calculate proportions

In [ ]:
df_hgmd_pt['mutype'].value_counts()

In [ ]:
# Merge and deduplicate
import pandas as pd

print(f"[INFO] Collecting all pathogenic variant...")
# Load all reviewed pathogenic ClinVar SNVs
df_clinvar_R = pd.read_csv("../dataset/a-middleware/ClinVar.reviewed.txt", sep='\t', usecols=['variant38', 'MC', 'CLNSIG'])
clinvar_R_pt = set(df_clinvar_R[(df_clinvar_R['CLNSIG'] == 'Pathogenic') | (df_clinvar_R['CLNSIG'] == 'Likely_pathogenic') | (df_clinvar_R['CLNSIG'] == 'Pathogenic/Likely_pathogenic')]['variant38'])

# Load all HGMD DM variants
hgmd_usecols = ['acc_num', 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38', 'amino', 'disease', 'tag', 'mutype', 'refseq', 'hgvs', 'codon']
df_hgmd = pd.read_csv("../dataset/x-source-data/HGMD_2023.3_allmut.csv", usecols=hgmd_usecols)
df_hgmd['pos_hg38'] = df_hgmd['pos_hg38'].astype('Int64') # Nullable integer type
df_hgmd['chrom_hg38'] = df_hgmd['chrom_hg38'].astype(str) 
nucleotides = {'A', 'C', 'T', 'G'}
chr_list = [str(i) for i in range(1, 23)] + ['X', 'Y']
df_hgmd_pt = df_hgmd[
	(df_hgmd['chrom_hg38'].isin(chr_list)) &
	(df_hgmd['tag'] == 'DM') &
	(df_hgmd['pos_hg38'].notna()) &
	(df_hgmd['ref_hg38'].isin(nucleotides)) &
	(df_hgmd['alt_hg38'].isin(nucleotides)) &
	(df_hgmd['ref_hg38'] != df_hgmd['alt_hg38'])
]
df_hgmd_pt['variant38'] = retrieve_variant(df_hgmd_pt, 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38')
hgmd_pt = set(df_hgmd_pt['variant38'])

# Merge pathogenic HGMD and ClinVar variants
all_pt = clinvar_R_pt | hgmd_pt
print(f"[INFO] Total pathogenic variants in ClinVar (P/LP) and HGMD (DM): {len(all_pt)}")

# Load each molecular-consequence type and take the union
# Non-coding
df_nc_pt = pd.read_csv("../dataset/a-middleware/noncoding/pathogenic.all.mutype.source.evidence.txt", sep='\t') # Merged HGMD and ClinVar result
nc_pt = set(df_nc_pt[df_nc_pt['evidence_level']!='DM?']['variant38']) # Exclude DM? consistently
print(f"[INFO] Pathogenic non-coding variants: {len(nc_pt)}")

# Missense
mis_clinvar_pt = set(pd.read_csv("../dataset/a-middleware/ClinVar.missense.pathogenic.txt", sep='\t', usecols=['variant38'])['variant38'])
mis_hgmd_pt = set(pd.read_csv("../dataset/a-middleware/HGMD.missense.dm.txt", sep='\t', usecols=['variant_hg38'])['variant_hg38'])
mis_pt = mis_clinvar_pt | mis_hgmd_pt
print(f"[INFO] Pathogenic missense variants: {len(mis_pt)}")

# Nonsense
nonsense_clinvar_pt = set(pd.read_csv("../dataset/a-middleware/ClinVar.nonsense.pathogenic.txt", sep='\t')['variant38'])
nonsense_hgmd_pt = set(pd.read_csv("../dataset/a-middleware/HGMD.nonsense.dm.txt", sep='\t')['variant_hg38'])
nonsense_pt = nonsense_clinvar_pt | nonsense_hgmd_pt
print(f"[INFO] Pathogenic nonsense variants: {len(nonsense_pt)}")

# Synonymous
syn_hgmd_pt = set(pd.read_csv("../dataset/a-middleware/synonymous/HGMD.synonymous.dm.txt", sep='\t')['variant_hg38'])
syn_clinvar_pt = set(pd.read_csv("../dataset/a-middleware/synonymous/ClinVar.synonymous.pathogenic.txt", sep='\t')['variant_hg38'])
syn_pt = syn_hgmd_pt | syn_clinvar_pt
print(f"[INFO] Pathogenic synonymous variants: {len(syn_pt)}")

# Remaining variants
remain = all_pt - nc_pt - mis_pt - nonsense_pt - syn_pt
print(f"[INFO] Remaining variants: {len(remain)}")

## 4.3 Plot with black text outlines

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

# Data
amout = {
    'Noncoding': 53177, 
    'Missense': 137874,
    'Synonymous': 1542,
    'Nonsense': 67911,
    'Other': 12931
}

# Pie-chart colors
colors = ['#000080', '#0000ff', "#5000f0", '#d6256d', '#f5f5f5']

# Calculate percentages
labels_raw = list(amout.keys())
sizes = list(amout.values())
total = sum(sizes)
labels = [
    f"{label}\n({size/total*100:.1f}%)"
    for label, size in zip(labels_raw, sizes)
]

# Synonymous explode
explode = [
    0.08 if label == 'Synonymous' else 0
    for label in labels_raw
]

# Plot
plt.figure(figsize=(6, 6))
wedges, texts = plt.pie(
    sizes,
    labels=labels,
    colors=colors,
    explode=explode,
    startangle=99,
    counterclock=False,
    labeldistance=0.6,
    wedgeprops={
        'linewidth': 0.5,
        'edgecolor': 'white'
    }
)

# ============ Use consistent white text with black outlines ============
for text in texts:
    text.set_color('white')  # Set consistently to white
    text.set_fontsize(17)
    text.set_fontweight(600)
    text.set_ha('center')
    # Add a black outline
    text.set_path_effects([
        path_effects.withStroke(
            linewidth=3,  # Outline width
            foreground='black'  # Outline color
        )
    ])

plt.axis('equal')
plt.tight_layout()
plt.show()

# 5 Figure 1(A) - Timeline

Increase in the number of pathogenic synonymous variants
- ClinVar does not record the original submission time, so LastEvaluated is used.
- For HGMD, Year is used and represents publication year. The HGMD field corresponding more closely to LastEvaluated would be new_date.

## 5.1 Process ClinVar data

In [ ]:
# clinvar
import pandas as pd

syn_df = pd.read_csv("../dataset/a-middleware/synonymous/ClinVar.synonymous.pathogenic.txt", sep='\t')
df_all = pd.read_csv("../dataset/x-source-data/variant_summary.txt", sep='\t', usecols=['Name', 'LastEvaluated'])
syn_df = syn_df.merge(df_all, how='left', left_on='Name', right_on='Name', suffixes=('', '_m'))

In [ ]:
# df_all contains both GRCh37 and GRCh38 records, so duplicates must be removed
syn_df.drop_duplicates(subset='Name', inplace=True)
syn_df.to_csv("../dataset/a-middleware/synonymous/ClinVar.synonymous.pathogenic.withtimeinfo.txt", sep='\t', index=False)

In [ ]:
# Convert to datetime
import pandas as pd
syn_df = pd.read_csv("../dataset/a-middleware/synonymous/ClinVar.synonymous.pathogenic.withtimeinfo.txt", sep='\t')

syn_df['LastEvaluated_dt'] = pd.to_datetime(
    syn_df['LastEvaluated'],
    format='%b %d, %Y',
    errors='coerce'
)

# 2. Extract year
syn_df['year'] = syn_df['LastEvaluated_dt'].dt.year
syn_df.to_csv("../dataset/a-middleware/synonymous/ClinVar.synonymous.pathogenic.withtimeinfo.txt", sep='\t', index=False)

## 5.2 Process HGMD data

In [ ]:
# hgmd
import pandas as pd

df_syn_hgmd = pd.read_csv("../dataset/a-middleware/synonymous/HGMD.synonymous.dm.txt",sep='\t', usecols=['variant_hg38'])
print(f"[INFO] Pathogenic sSNVs from HGMD(DM): {len(df_syn_hgmd)}")

# hgmd timeline
df_hgmd = pd.read_csv("../dataset/x-source-data/HGMD_2023.3_allmut.csv", usecols=['chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38', 'year'])
df_hgmd = df_hgmd[df_hgmd['pos_hg38'].notna()]
df_hgmd['pos_hg38'] = df_hgmd['pos_hg38'].astype(int)
df_hgmd['year'] = df_hgmd['year'].astype(int)
df_hgmd['variant_hg38'] = retrieve_variant(df_hgmd, 'chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38')
df_hgmd.drop(['chrom_hg38', 'pos_hg38', 'ref_hg38', 'alt_hg38'], axis=1, inplace=True)
df_hgmd = df_hgmd[df_hgmd['variant_hg38'].isin(df_syn_hgmd['variant_hg38'])]
# NOTE If one publication has multiple records, use the earliest record when merging
df_hgmd = df_hgmd.sort_values('year').drop_duplicates('variant_hg38', keep='first')

# merge
df_syn_hgmd = df_syn_hgmd.merge(df_hgmd, how='left', left_on='variant_hg38', right_on='variant_hg38', suffixes=('', '_m'))
print(f"[INFO] After merge timeline: {len(df_syn_hgmd)}")
df_syn_hgmd.head(2)
df_syn_hgmd.to_csv("../dataset/a-middleware/synonymous/HGMD.synonymous.dm.with.timeline.txt", sep='\t', index=False)

## 5.3 Merge and plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Load separately
root_path = "../dataset/a-middleware/synonymous/"
df_syn_hgmd_time = pd.read_csv(f"{root_path}/HGMD.synonymous.dm.with.timeline.txt", sep='\t')
print(f"[INFO] HGMD synonymous pathogenic Data: {len(df_syn_hgmd_time)}")
df_syn_hgmd_time = df_syn_hgmd_time[df_syn_hgmd_time['year'].notna()]
print(f"[INFO] HGMD synonymous pathogenic Data with time information: {len(df_syn_hgmd_time)}")

df_syn_clinvar_time = pd.read_csv(f"{root_path}/ClinVar.synonymous.pathogenic.withtimeinfo.txt", sep='\t', usecols=['variant_hg38', 'year'])
print(f"[INFO] ClinVar synonymous pathogenic Data: {len(df_syn_clinvar_time)}")
df_syn_clinvar_time = df_syn_clinvar_time[df_syn_clinvar_time['year'].notna()]
print(f"[INFO] ClinVar synonymous pathogenic Data with time information: {len(df_syn_clinvar_time)}")

# Merge
df_syn_time = pd.concat([df_syn_hgmd_time, df_syn_clinvar_time], ignore_index=True)
# NOTE If one publication has multiple records, use the earliest record when merging
df_syn_time = df_syn_time.sort_values('year').drop_duplicates('variant_hg38', keep='first')

# 3. Restrict the time range to 2023 or earlier because ClinVar 2025.04 and HGMD 2023.3 cover different periods; use the earlier endpoint.
df_syn_time = df_syn_time[df_syn_time['year'] < 2023]

# 4. Count records per year without filling missing years
year_counts = (
    df_syn_time['year']
    .value_counts()
    .sort_index()
)

# ============ Control marker density ============
plot_every_n_years = 1  # Use 1 for annual markers or 5 for markers every five years

# Generate x-axis ticks every five years from the earliest year through 2023
xticks_5yr = list(range(
    int(year_counts.index.min()),  # 1983
    2024,  # Through 2023
    5
))

# 7. Plot
plt.figure(figsize=(5.5, 4))
plot_color = "#0000ff61"
line_color = "#d6256d"

# Plot annual increments for years with data
plt.plot(
    year_counts.index,
    year_counts.values,
    marker='o',
    linewidth=1,
    markersize=10,
    color=line_color,
    markerfacecolor=plot_color,
    markeredgecolor=plot_color,
    markeredgewidth=1.5,
)

# Optional: show markers only for selected years
if plot_every_n_years > 1:
    plt.clf()
    plt.plot(
        year_counts.index,
        year_counts.values,
        linewidth=1,
        color=line_color,
    )
    # Mark selected years only
    marker_years = year_counts[year_counts.index % plot_every_n_years == 0]
    plt.scatter(marker_years.index, marker_years.values, 
               color=line_color, s=80, zorder=5,
               facecolor=plot_color, edgecolor=plot_color, linewidth=1.5)

mono_color = '#222'

plt.xlabel('Year', fontsize=14, color=mono_color)
plt.ylabel('Annual count', fontsize=14, color=mono_color)

plt.tick_params(axis='x', labelsize=12, width=1.3, color=mono_color, labelcolor=mono_color)
plt.tick_params(axis='y', labelsize=12, width=1.3, color=mono_color, labelcolor=mono_color)

# Set plot-spine widths
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color(mono_color)

# ============ Set the x-axis range and add horizontal margins ============
# Use 5% of the total range as the margin
x_min = year_counts.index.min()
x_max = 2023
x_margin = (x_max - x_min) * 0.05  # 5% margin

ax.set_xlim(x_min - x_margin, x_max + x_margin)

# Set x-axis ticks
plt.xticks(xticks_5yr)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\n[Summary statistics]")
print(f"Data year range: {year_counts.index.min()} - {year_counts.index.max()}")
print(f"X-axis display range: {x_min - x_margin:.1f} - {x_max + x_margin:.1f}")
print(f"Total records: {year_counts.sum()}")
print(f"Years with data: {len(year_counts)}")